# SD3.5 LoRA Training — Kaggle Runner

Full pipeline: `raw datasets → build release → validate → train → export`

**All logic lives in Python modules. This notebook only calls them.**  
Hard-fails at Cell 06, 07, or 08 if any contract is not met.

**Smoke run (first time):** 01 → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09a → 10  
**Full train:** 01–08 → 09a → 09b → 09c → 10 → 11

## 01. Install Dependencies

In [ ]:
!pip install -q 'diffusers==0.31.0' 'transformers>=4.44.0' 'accelerate>=0.33.0' 'datasets>=2.20.0' 'safetensors>=0.4.3' 'bitsandbytes>=0.43.0' 'peft>=0.12.0' 'pillow>=10.0.0' 'imagehash>=4.3.1' 'pandas>=2.0.0' 'pyarrow>=14.0.0' tqdm matplotlib

import diffusers, datasets, accelerate, safetensors
print(f'diffusers:   {diffusers.__version__}')
print(f'datasets:    {datasets.__version__}')
print(f'accelerate:  {accelerate.__version__}')
print(f'safetensors: {safetensors.__version__}')

## 02. Clone Repo + Setup Path

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO_URL = 'https://github.com/BDT-17/VIN.git'
REPO_DIR = Path('/kaggle/working/VIN')

if REPO_DIR.exists():
    out = subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], capture_output=True, text=True)
    print(out.stdout.strip() or 'Already up to date.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

sha = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    capture_output=True, text=True,
).stdout.strip()
print(f'Git SHA: {sha}')
print(f'Python:  {sys.version}')

# Two import styles are used by the code, so TWO paths are needed:
#   1. from LoRA.data.pipeline import LoRAPipeline   -> needs REPO_DIR (parent of LoRA package)
#   2. from sd35_config import *  (inside LoRA/*.py)  -> needs REPO_DIR/LoRA on the path
# LORA_DIR MUST come first: the repo ROOT also has a sd35_config.py (augmentation config,
# without LoRA-training names). Putting LORA_DIR first makes bare `import sd35_config`
# resolve to LoRA/sd35_config.py which defines LORA_TRAINING_*.
PROJECT_DIR = REPO_DIR
LORA_DIR    = REPO_DIR / 'LoRA'
assert LORA_DIR.exists(), f'LoRA package not found: {LORA_DIR}'
for p in (str(PROJECT_DIR), str(LORA_DIR)):
    if p in sys.path:
        sys.path.remove(p)
sys.path = [str(LORA_DIR), str(PROJECT_DIR)] + sys.path
print(f'PROJECT_DIR: {PROJECT_DIR}')
print(f'LORA_DIR:    {LORA_DIR}')

# Download pinned Diffusers DreamBooth LoRA SD3 training script
TRAIN_SCRIPT = Path('/kaggle/working/train_dreambooth_lora_sd3.py')
if not TRAIN_SCRIPT.exists():
    subprocess.run([
        'wget', '-q',
        'https://raw.githubusercontent.com/huggingface/diffusers/v0.31.0/examples/dreambooth/train_dreambooth_lora_sd3.py',
        '-O', str(TRAIN_SCRIPT),
    ], check=True)
    print(f'Downloaded: {TRAIN_SCRIPT}')
else:
    print(f'Exists: {TRAIN_SCRIPT}')

os.environ['SD35_LORA_TRAIN_SCRIPT'] = str(TRAIN_SCRIPT)
print('SD35_LORA_TRAIN_SCRIPT set.')

## 03. GPU Preflight

In [ ]:
import torch

assert torch.cuda.is_available(), 'No CUDA GPU. Attach a GPU accelerator before continuing.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU:     {gpu_name}')
print(f'VRAM:    {vram_gb:.1f} GB')
print(f'CUDA:    {torch.version.cuda}')
print(f'PyTorch: {torch.__version__}')

if vram_gb < 14:
    print(f'WARNING: VRAM {vram_gb:.1f} GB < 14 GB recommended for fp16 batch=1 at 512px')

## 04. Verify Kaggle Dataset Mounts

In [ ]:
import yaml
from pathlib import Path

SOURCES_YAML = PROJECT_DIR / 'LoRA' / 'data' / 'sources.yaml'
assert SOURCES_YAML.exists(), f'sources.yaml not found: {SOURCES_YAML}'

with open(SOURCES_YAML) as f:
    _src_cfg = yaml.safe_load(f)

# Candidate mount BASE paths per source. The first base that satisfies the
# parser's expected sub-structure is used. Covers both Kaggle mount forms:
#   /kaggle/input/<slug>            (standard "Add Data")
#   /kaggle/input/datasets/<user>/<slug>
CANDIDATES = {
    'citypersons': [   # YOLO: base must contain <split>/images
        '/kaggle/input/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir',
        '/kaggle/input/datasets/muttahirulislam/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir',
        '/kaggle/input/citypersons-canonical',
    ],
    'mot17_02': [      # MOT: base must contain MOT17-02-FRCNN/img1
        '/kaggle/input/mot17-02-fcrnn',
        '/kaggle/input/datasets/kyoru4444/mot17-02-fcrnn',
    ],
    'human_detection': [  # classification: base must contain "0" and "1"
        '/kaggle/input/human-detection-dataset/human detection dataset',
        '/kaggle/input/datasets/constantinwerner/human-detection-dataset/human detection dataset',
        '/kaggle/input/human-detection-dataset',
    ],
}

def _resolve_yolo(src, bases):
    """Pick a base that has <train>/images, and detect valid vs val naming."""
    for base in bases:
        b = Path(base)
        if (b / 'train' / 'images').exists():
            # detect val folder name
            val_name = 'valid' if (b / 'valid' / 'images').exists() else (
                'val' if (b / 'val' / 'images').exists() else None)
            splits = {'train': 'train/images'}
            labels = {'train': 'train/labels'}
            if val_name:
                splits['val'] = f'{val_name}/images'
                labels['val'] = f'{val_name}/labels'
            if (b / 'test' / 'images').exists():
                splits['test'] = 'test/images'
                labels['test'] = 'test/labels'
            else:
                print(f"  ⚠  {src['source_id']}: no test/images — benchmark_lock test split will be empty")
            src['kaggle_mount'] = str(b)
            src['splits'] = splits
            src['label_dirs'] = labels
            return str(b)
    return None

def _resolve_mot(src, bases):
    seq = src.get('sequence_dir', 'MOT17-02-FRCNN/img1')
    for base in bases:
        if (Path(base) / seq).exists():
            src['kaggle_mount'] = str(base)
            return str(base)
    return None

def _resolve_classification(src, bases):
    pos, neg = src.get('positive_dir', '1'), src.get('negative_dir', '0')
    for base in bases:
        b = Path(base)
        if (b / pos).exists() or (b / neg).exists():
            src['kaggle_mount'] = str(b)
            return str(b)
    return None

RESOLVERS = {'yolo': _resolve_yolo, 'mot': _resolve_mot, 'classification_folders': _resolve_classification}

all_ok = True
for src in _src_cfg['sources']:
    bases = CANDIDATES.get(src['source_id'], [src['kaggle_mount']])
    resolved = RESOLVERS[src['parser']](src, bases)
    if resolved:
        print(f"  ✓  {src['source_id']:16s} [{src['parser']}] -> {resolved}")
    else:
        print(f"  ✗ MISSING  {src['source_id']:16s} [{src['parser']}] — tried:")
        for c in bases:
            print(f"        {c}")
        all_ok = False

assert all_ok, 'Could not resolve one or more dataset mounts. Check Add Data, or edit CANDIDATES above.'

# Write resolved config; Cell 05 reads this instead of the repo default
SOURCES_YAML_RESOLVED = Path('/kaggle/working/sources_resolved.yaml')
with open(SOURCES_YAML_RESOLVED, 'w') as f:
    yaml.safe_dump(_src_cfg, f, sort_keys=False, allow_unicode=True)
print(f'\nResolved config -> {SOURCES_YAML_RESOLVED}')

# SD3.5 model: local Kaggle mount takes priority, otherwise HuggingFace download
_sd35_local = Path('/kaggle/input/stable-diffusion-3-5-medium')
if _sd35_local.exists():
    SD35_MODEL_PATH = str(_sd35_local)
    print(f'  ✓  sd35_model (local): {_sd35_local}')
else:
    SD35_MODEL_PATH = 'stabilityai/stable-diffusion-3.5-medium'
    print(f'  ~  sd35_model: not mounted — will download from HuggingFace (needs HF_TOKEN in Cell 04b)')

## 04b. HuggingFace Login + SD3.5 Model Path

Skip if SD3.5 is mounted locally (Cell 04 printed `sd35_model (local)`).

In [ ]:
import os

if SD35_MODEL_PATH.startswith('stabilityai/'):
    # Need HF_TOKEN to download the model during training
    hf_token = None
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        hf_token = os.environ.get('HF_TOKEN')

    if not hf_token:
        raise RuntimeError(
            'HF_TOKEN not found.\n'
            'Option A: Add a Kaggle secret named HF_TOKEN (Settings → Secrets).\n'
            'Option B: Mount the SD3.5 model dataset so /kaggle/input/stable-diffusion-3-5-medium exists.'
        )
    from huggingface_hub import login
    login(token=hf_token)
    print('HuggingFace login OK — model will be downloaded during training.')
else:
    print(f'Using local SD3.5 model at: {SD35_MODEL_PATH}')

# Patch SD35_MODEL_ID on the BARE sd35_config module — this is the exact module
# object that LoRA/sd35_lora_training.py reads via `from sd35_config import *`.
# (Do NOT use `import LoRA.sd35_config`; that is a different module object.)
import sd35_config as _cfg
_cfg.SD35_MODEL_ID = SD35_MODEL_PATH
print(f'SD35_MODEL_ID = {_cfg.SD35_MODEL_ID}')

## 05. Build Dataset Release

In [ ]:
from pathlib import Path
from LoRA.data.pipeline import LoRAPipeline

# Use the runtime-resolved config from Cell 04 (correct mount paths for THIS
# Kaggle session), not the repo default which assumes canonical slugs.
WORKING_DIR = Path('/kaggle/working/vin_data')
assert SOURCES_YAML_RESOLVED.exists(), 'Run Cell 04 first — SOURCES_YAML_RESOLVED missing.'

pipeline    = LoRAPipeline(config_path=SOURCES_YAML_RESOLVED, working_dir=WORKING_DIR)
RELEASE_DIR = pipeline.run_full_pipeline()
print(f'\nRelease directory: {RELEASE_DIR}')

## 06. Validate Release (Hard Fail)

In [ ]:
from LoRA.data.validate import validate_release

result = validate_release(RELEASE_DIR)
print(f"Valid:  {result['valid']}")
print(f"  train samples: {result['stats'].get('train_count', 0)}")
print(f"  val   samples: {result['stats'].get('val_count', 0)}")

if result.get('warnings'):
    for w in result['warnings']:
        print(f'  \u26a0  {w}')

assert result['valid'], (
    'RELEASE VALIDATION FAILED:\n'
    + '\n'.join(f'  - {e}' for e in result.get('errors', []))
)

## 07. ImageFolder Contract Test (Hard Fail)

In [ ]:
# Verifies datasets.load_dataset('imagefolder', ...) produces 'image' and 'text' columns.
# Trainer requires --image_column image --caption_column text.
from LoRA.sd35_lora_training import test_imagefolder_contract
test_imagefolder_contract(RELEASE_DIR)

## 08. Trainer Dry Run

In [ ]:
# Builds and prints the accelerate launch command without running it.
from LoRA.sd35_lora_training import run_lora_training

dry = run_lora_training(dataset_release=str(RELEASE_DIR), dry_run=True)
print('\nDry run OK. Command saved to:', dry['command_path'])

## 09a. Smoke Train (50 steps)

Proves pipeline + adapter export/load work end-to-end.  
Run this before committing to the 1000-step full train.

In [ ]:
import importlib
import sd35_config as _cfg          # bare module — same object the trainer reads
from pathlib import Path

_cfg.LORA_TRAINING_MAX_TRAIN_STEPS     = 50
_cfg.LORA_TRAINING_OUTPUT_DIR          = Path('/kaggle/working/lora_smoke')
_cfg.LORA_TRAINING_CHECKPOINTING_STEPS = 50

import LoRA.sd35_lora_training as _training
importlib.reload(_training)          # re-runs `from sd35_config import *`, picks up patches

smoke_result = _training.run_lora_training(
    dataset_release=str(RELEASE_DIR),
    dry_run=False,
)
print('\nSmoke train complete.')
print('Adapter:', smoke_result.get('adapter_path'))
print('Run ID: ', smoke_result.get('training_run_id'))

## 09b. Full Train (1000 steps)

Only run after Cell 09a passes without error.

In [ ]:
import importlib
import sd35_config as _cfg          # bare module — same object the trainer reads
from pathlib import Path

_cfg.LORA_TRAINING_MAX_TRAIN_STEPS     = 1000
_cfg.LORA_TRAINING_OUTPUT_DIR          = Path('/kaggle/working/sd35m-pedestrian-v1')
_cfg.LORA_TRAINING_CHECKPOINTING_STEPS = 250

import LoRA.sd35_lora_training as _training
importlib.reload(_training)          # re-runs `from sd35_config import *`, picks up patches

train_result = _training.run_lora_training(
    dataset_release=str(RELEASE_DIR),
    dry_run=False,
)
print('Adapter:', train_result.get('adapter_path'))
print('PT:     ', train_result.get('pt_path'))

## 09c. Training Metrics (run after 09b)

In [ ]:
import json
from pathlib import Path

REPORTS_DIR = Path(train_result['training_reports_dir'])
assert (REPORTS_DIR / 'summary.json').exists(), f'summary.json missing: {REPORTS_DIR}'

with open(REPORTS_DIR / 'summary.json') as f:
    s = json.load(f)

print(f"Run ID:    {s.get('run_id')}")
print(f"Steps:     {s.get('total_steps')}")
print(f"Duration:  {s.get('total_seconds', 0) / 60:.1f} min")
loss = s.get('loss', {})
print(f"Loss:      first={loss.get('first')}  last={loss.get('last')}  min={loss.get('min')}")
tp = s.get('throughput', {})
print(f"Throughput: {tp.get('steps_per_second')} steps/sec")

curve = REPORTS_DIR / 'loss_curve.png'
if curve.exists():
    from IPython.display import Image, display
    display(Image(str(curve)))

## 10. Verify Adapter Loadable

In [ ]:
from LoRA.sd35_lora_training import verify_adapter_loadable
from pathlib import Path

# Works with both smoke_result (09a) and train_result (09b)
_result = locals().get('train_result') or smoke_result
adapter_path = Path(_result['adapter_path'])

v = verify_adapter_loadable(adapter_path)
print('Adapter verification:', v)
assert v['loadable'], f"Adapter not loadable: {v.get('error')}"
print(f"✓ Adapter loadable ({v.get('key_count', '?')} keys)")

## 11. Zip Artifacts for Download

In [ ]:
import zipfile
from pathlib import Path

_result    = locals().get('train_result') or smoke_result
OUTPUT_DIR = Path(_result['adapter_path']).parent
ZIP_PATH   = Path('/kaggle/working/sd35m-pedestrian-v1-artifacts.zip')

ARTIFACT_NAMES = [
    'pytorch_lora_weights.safetensors',
    'pytorch_lora_weights.pt',
    'training_config.json',
    'training_provenance.json',
    'dataset_provenance.json',
    'train_command.json',
    'pip_freeze.txt',
    'gpu_info.json',
    'validation_prompts.json',
    'adapter_verification.json',
]

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for name in ARTIFACT_NAMES:
        p = OUTPUT_DIR / name
        if p.exists():
            zf.write(p, name)
            print(f'  + {name}')
        else:
            print(f'  - MISSING: {name}')

    reports_dir = Path(_result.get('training_reports_dir', ''))
    if reports_dir.exists():
        for fp in sorted(reports_dir.iterdir()):
            zf.write(fp, f'reports/training/{fp.name}')
            print(f'  + reports/training/{fp.name}')

print(f'\n\u2713 Zipped: {ZIP_PATH}  ({ZIP_PATH.stat().st_size / (1024**2):.1f} MB)')